[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/solutions/46_patchify_latents_solution.ipynb)

# 🟡 Solution: Patchify / Unpatchify Latents

Reference solution for `patchify_latents`.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

class PatchifyLatents:
    def __init__(self, patch_size):
        if isinstance(patch_size, tuple):
            self.patch_h, self.patch_w = patch_size
        else:
            self.patch_h = self.patch_w = patch_size

    def patchify(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        pH, pW = self.patch_h, self.patch_w
        if H % pH != 0 or W % pW != 0:
            raise ValueError("image height and width must be divisible by patch_size")
        h, w = H // pH, W // pW
        x = x.view(B, C, h, pH, w, pW)
        x = x.permute(0, 2, 4, 1, 3, 5).contiguous()
        return x.view(B, h * w, C * pH * pW)

    def unpatchify(self, tokens: torch.Tensor, image_size, channels: int) -> torch.Tensor:
        H, W = image_size
        pH, pW = self.patch_h, self.patch_w
        B, N, patch_dim = tokens.shape
        h, w = H // pH, W // pW
        if N != h * w or patch_dim != channels * pH * pW:
            raise ValueError("tokens do not match image_size/channels/patch_size")
        x = tokens.view(B, h, w, channels, pH, pW)
        x = x.permute(0, 3, 1, 4, 2, 5).contiguous()
        return x.view(B, channels, H, W)


In [ ]:
# Verify
x = torch.arange(2 * 3 * 4 * 4).float().view(2, 3, 4, 4)
patcher = PatchifyLatents(2)
tokens = patcher.patchify(x)
print(tokens.shape)
print(torch.equal(patcher.unpatchify(tokens, image_size=(4, 4), channels=3), x))


In [ ]:
# Run judge
from torch_judge import check
check('patchify_latents')
